# Deepfake Detector — Full Training on Colab GPU

**Setup:**
1. Upload your project zip to Google Drive (or clone from GitHub)
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells top to bottom

Outputs (checkpoints + artifacts) are saved back to your Drive automatically.

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch runtime!')

In [ ]:
# ── Cell 2: Mount Google Drive (for saving outputs) ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Where to save checkpoints/artifacts back to Drive after training
DRIVE_OUT = '/content/drive/MyDrive/deepfake_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Drive mounted. Outputs will be saved to:', DRIVE_OUT)

In [ ]:
# ── Cell 3: Extract dataset + artifacts ─────────────────────────────────────
import os

# dataset.zip and artifacts.zip are already uploaded to /content/
!unzip -q /content/dataset.zip   -d /content/project/dataset_extracted
!unzip -q /content/artifacts.zip -d /content/project/artifacts_extracted

# Clone your project code from GitHub (replace with your repo URL)
# OR manually upload your src/ folder zip
# Option A — GitHub (recommended):
GITHUB_REPO = ''  # e.g. 'https://github.com/yourname/deepfake-detection'

if GITHUB_REPO:
    !git clone "{GITHUB_REPO}" /content/project/code
    PROJECT_ROOT = '/content/project/code'
else:
    # Option B — upload your src.zip to /content/ and unzip it
    if os.path.exists('/content/src.zip'):
        !unzip -q /content/src.zip -d /content/project/
    PROJECT_ROOT = '/content/project'

# Find dataset folder (handles nested zip structure)
def find_dir(base, name):
    for root, dirs, _ in os.walk(base):
        if name in dirs:
            return os.path.join(root, name)
    return None

dataset_dir = find_dir('/content/project/dataset_extracted', 'real') or \
              find_dir('/content/project/dataset_extracted', 'fake')
if dataset_dir:
    dataset_dir = os.path.dirname(dataset_dir)  # parent of real/fake
    print('Dataset found at:', dataset_dir)
else:
    dataset_dir = '/content/project/dataset_extracted'
    print('Warning: could not auto-detect dataset structure, using:', dataset_dir)

# Symlink dataset into project root so src/config.py finds it
link = os.path.join(PROJECT_ROOT, 'dataset')
if not os.path.exists(link):
    os.symlink(dataset_dir, link)

# Symlink artifacts
artifacts_src = '/content/project/artifacts_extracted'
# find actual artifacts folder
for root_d, dirs, files in os.walk(artifacts_src):
    if 'metrics.json' in files or 'history.json' in files:
        artifacts_src = root_d
        break
alink = os.path.join(PROJECT_ROOT, 'artifacts')
if not os.path.exists(alink):
    os.symlink(artifacts_src, alink)

print('Project root:', PROJECT_ROOT)
print('Contents:', os.listdir(PROJECT_ROOT))

real_imgs = len(list(__import__('pathlib').Path(dataset_dir).glob('real/*.jpg')))
fake_imgs = len(list(__import__('pathlib').Path(dataset_dir).glob('fake/*.jpg')))
print(f'Images: {real_imgs} real, {fake_imgs} fake')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
!pip install -q pytorchcv albumentations
print('Dependencies installed.')

In [ ]:
# ── Cell 5: Configure paths & hyperparameters ────────────────────────────────
import sys
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# ── Training config (edit as needed) ────────────────────────────────────────
EPOCHS_P1       = 10    # frozen backbone — train head only
EPOCHS_P2       = 40    # fine-tune last stages + head
BATCH_SIZE      = 32    # increase to 64 if you have an A100
MAX_PER_CLASS   = None  # None = use ALL images; set e.g. 1000 for a faster test run
USE_ENSEMBLE    = True  # XceptionNet + EfficientNet
RESUME_FROM     = None  # set to '/content/drive/MyDrive/deepfake_outputs/checkpoints/resume.pt' to continue

print(f'Config: P1={EPOCHS_P1} P2={EPOCHS_P2} batch={BATCH_SIZE} ensemble={USE_ENSEMBLE}')
print(f'Dataset: {PROJECT_ROOT}/dataset')

In [ ]:
# ── Cell 6: Patch config for Colab paths & batch size ────────────────────────
from src import config as cfg
from pathlib import Path

cfg.BATCH_SIZE   = BATCH_SIZE
cfg.NUM_WORKERS  = 2          # Colab supports multiprocessing
cfg.EPOCHS_PHASE1 = EPOCHS_P1
cfg.EPOCHS_PHASE2 = EPOCHS_P2

# Verify dataset exists
real_count = len(list((cfg.DATA_DIR / 'real').glob('*.jpg')))
fake_count = len(list((cfg.DATA_DIR / 'fake').glob('*.jpg')))
print(f'Dataset: {real_count} real, {fake_count} fake images')
assert real_count > 0 and fake_count > 0, 'Dataset not found! Check PROJECT_ROOT path.'

In [ ]:
# ── Cell 7: Run training ─────────────────────────────────────────────────────
from src.train import main as train_main

args = [
    f'--epochs-p1={EPOCHS_P1}',
    f'--epochs-p2={EPOCHS_P2}',
    f'--batch-size={BATCH_SIZE}',
]

if not USE_ENSEMBLE:
    args.append('--no-ensemble')

if MAX_PER_CLASS is not None:
    args.append(f'--max-per-class={MAX_PER_CLASS}')

if RESUME_FROM is not None:
    # copy resume checkpoint to local checkpoints dir first
    import shutil
    shutil.copy(RESUME_FROM, str(cfg.CHECKPOINT_DIR / 'resume.pt'))
    args.append(f'--resume={cfg.CHECKPOINT_DIR}/resume.pt')

print('Running:', 'python -m src.train', ' '.join(args))
train_main(args)

In [ ]:
# ── Cell 8: Save outputs back to Google Drive ────────────────────────────────
import shutil, json

# Copy checkpoints
shutil.copytree(str(cfg.CHECKPOINT_DIR), os.path.join(DRIVE_OUT, 'checkpoints'), dirs_exist_ok=True)

# Copy artifacts
shutil.copytree(str(cfg.ARTIFACTS_DIR), os.path.join(DRIVE_OUT, 'artifacts'), dirs_exist_ok=True)

# Print final metrics
metrics_path = cfg.ARTIFACTS_DIR / 'metrics.json'
if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    print('\n=== Final Results ===')
    print(f"Test Accuracy : {m['test_accuracy']*100:.2f}%")
    print(f"Best Val Acc  : {m['best_val_accuracy']*100:.2f}%")
    print(f"ROC AUC       : {m['test_roc_auc_fake']:.4f}")
    print(f"\nClassification Report:\n{m.get('classification_report','')}")

print(f'\nAll outputs saved to: {DRIVE_OUT}')

In [ ]:
# ── Cell 9: Show training curves (optional) ──────────────────────────────────
from IPython.display import Image as IPImage, display
import os

curves = str(cfg.ARTIFACTS_DIR / 'training_curves.png')
cm     = str(cfg.ARTIFACTS_DIR / 'figures' / 'confusion_matrix_test.png')

for path, title in [(curves, 'Training Curves'), (cm, 'Confusion Matrix')]:
    if os.path.exists(path):
        print(title)
        display(IPImage(path))